# WearableQA: multimodal-phenotype questions, explored and split by device

This notebook:
1. loads WearableQA and selects the **multimodal phenotype** questions;
2. plots the class distribution, the main parameters and the missing values;
3. splits each question's data into **private views per device or site** (watch, phone, ring, clinic) with
   `utility/wqa_split.py`;
4. shows how the data is split, through examples.

**Data.** `code/data/wearableqa/WearableQA_raw.json` (39 MB), from
https://github.com/facebookresearch/WearableQA. Licence: CC BY-NC 4.0 plus extra terms (no re-identification, no
clinical use, research and benchmarking only).

**Run.** From `code/`: `source .venv/bin/activate`, then `pip install pandas matplotlib jupyter` (once), then
`jupyter lab src/wqa_phenotype_explore.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from collections import Counter
from pathlib import Path

%pip install matplotlib
%pip install pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CODE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(CODE_DIR / "src"))
from utility import wqa_split as ws

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
CLASS_COLOURS = {"CH": "#2a9d8f", "CA": "#e76f51", "D-LF": "#f4a261", "D-FA": "#8d6cab"}
print("code dir:", CODE_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 16.9 MB/s  0:00:00 eta 0:00:01


## 1. Load the dataset and select the multimodal-phenotype questions

In [4]:
raw = ws.load_raw(ws.default_raw_path(CODE_DIR))
print(raw["description"])
print(f"{len(raw['mcqs'])} questions, {len(raw['user_histories'])} users")
print("question types:", Counter(q["question_type"] for q in raw["mcqs"]).most_common())

questions = ws.select_questions(raw, "multimodal_phenotype")
users = {q["user_id"] for q in questions}
print(f"\nmultimodal phenotype: {len(questions)} questions from {len(users)} users")
print("stems:", Counter(q["stem"] for q in questions))
print("signal/grounding:", Counter((q["signal"], q["grounding"]) for q in questions))
print("window sizes:", Counter(q["window_size"] for q in questions))
print("questions with masked metrics:", sum(bool(q.get("full_dropped_metrics")) for q in questions))
print("questions without cohort reference:", sum(q["no_cohort"] for q in questions))
print("questions with a blood panel:", sum(bool(q["context"].get("blood_panel")) for q in questions))

WearableQA v2.3 — wearable + blood biomarker multiple-choice QA benchmark.
4084 questions, 200 users
question types: [('signed_correlation', 774), ('excursion_count', 600), ('recovery_time', 447), ('prognostic_prediction', 445), ('cross_signal_prediction', 322), ('strongest_pair', 270), ('trend_shape', 235), ('anomaly_episodes', 212), ('health_recommendation', 124), ('fitness_prediction', 115), ('cohort_relative_risk', 108), ('strongest_triplet', 106), ('signal_concordance', 98), ('signal_summary', 80), ('multimodal_phenotype', 75), ('differential_diagnosis', 73)]

multimodal phenotype: 75 questions from 50 users
stems: Counter({"Based on this user's most recent 28-day wearable record and the available blood markers, which phenotype label best characterizes this individual?": 75})
signal/grounding: Counter({('cross', 'literature'): 75})
window sizes: Counter({28: 75})
questions with masked metrics: 0
questions without cohort reference: 0
questions with a blood panel: 75


In [ ]:
# One question as stored in the file (options with their class codes)
q = questions[0]
print("id:", q["id"], "| user:", q["user_id"], "| window ends:", q["end_date"])
print("query:", q["stem"])
codes = ws.option_classes(q)
for letter, text in q["options"].items():
    mark = "*" if letter == q["gt_letter"] else " "
    print(f"  {letter}{mark} [{codes[letter]:4s}] {text[:110]}")
print("demographics:", q["context"]["demographics"])
print("blood panel:", q["context"]["blood_panel"])
print("keys:", list(q.keys()))

id: Lit_user_1_2021-06-12_3 | user: user_1 | window ends: 2021-06-12
query: Based on this user's most recent 28-day wearable record and the available blood markers, which phenotype label best characterizes this individual?
  A* [CH  ] concordant healthy phenotype — favourable wearable fitness/activity together with a favourable measured metabo
  B  [T2D ] established type 2 diabetes phenotype with glycaemic dysregulation at the diagnostic threshold
  C  [CA  ] concordant adverse phenotype — low wearable fitness/activity together with an adverse measured metabolic-blood
  D  [INF ] acute infection phenotype reflecting a transient inflammatory state during the window
  E  [D-LF] discordant phenotype — low wearable fitness/activity BUT a favourable measured metabolic-blood profile (metabo
  F  [BMI ] the phenotype is attributable entirely to body-mass index, independent of the wearable and blood data
  G  [ATH ] athletic elite-endurance baseline well above population fitness norms
  H  [D

In [ ]:
# The wearable data is not inside the question: it is stored once per user in raw["user_histories"],
# and each question points into it through user_id and end_date.
hist = raw["user_histories"][q["user_id"]]
print(f"user history: {len(hist)} days, {hist[0]['date']} -> {hist[-1]['date']}")
print("masked for this question (full, window):", ws.question_masks(q))
q_rows = ws.question_rows(q, raw)  # 28-day window ending at end_date, masks applied
print(f"rows in the 28-day window: {len(q_rows)}")
display(pd.DataFrame(q_rows).set_index("date").tail(7))

## 2. Distributions

### 2a. Classes

The 10 options cover the 10 phenotype classes once each, in a shuffled order. Only the four 2 x 2 classes ever
appear as the correct answer.

In [ ]:
df = pd.DataFrame(ws.feature_rows(questions, raw, window_days=28))
names = ws.phenotype_class_names()
counts = df["gold_class"].value_counts().reindex(list(names), fill_value=0)
display(pd.DataFrame({"class": [names[c] for c in counts.index], "n_gold": counts.values}, index=counts.index))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
counts.plot.bar(ax=axes[0], color=[CLASS_COLOURS.get(c, "grey") for c in counts.index])
axes[0].set_title("Gold class (all 10 option classes)")
axes[0].set_ylabel("questions")
df["gold_letter"].value_counts().sort_index().plot.bar(ax=axes[1], color="grey")
axes[1].set_title("Gold answer letter")
plt.tight_layout()

### 2b. Users and demographics

In [ ]:
per_user = df.groupby("user_id").size()
users_df = df.drop_duplicates("user_id")
print("questions per user:", per_user.describe().round(2).to_dict())
print("does a user's gold class change across their questions?",
      (df.groupby("user_id")["gold_class"].nunique() > 1).sum(), "users with more than one class")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
per_user.value_counts().sort_index().plot.bar(ax=axes[0], color="grey")
axes[0].set_title("questions per user")
users_df["age"].plot.hist(ax=axes[1], bins=15, color="grey")
axes[1].set_title("age (one row per user)")
users_df["sex"].value_counts().plot.bar(ax=axes[2], color="grey")
axes[2].set_title("sex (one row per user)")
data = [df.loc[df.gold_class == c, "bmi"].dropna() for c in ws.core_classes()]
axes[3].boxplot(data, tick_labels=ws.core_classes())
axes[3].set_title("BMI by gold class")
plt.tight_layout()

### 2c. Wearable parameters by class

28-day window means per question, with the cohort median (dashed) where the dataset gives one.

In [ ]:
metrics_to_plot = ["rhr", "steps", "hrv", "vo2_max", "sleep_duration", "exercise_count", "active_burn",
                   "avg_stress_level"]
cohort = raw["cohort_reference"]
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, m in zip(axes.flat, metrics_to_plot):
    data = [df.loc[df.gold_class == c, f"mean_{m}"].dropna() for c in ws.core_classes()]
    ax.boxplot(data, tick_labels=[f"{c}\n(n={len(d)})" for c, d in zip(ws.core_classes(), data)])
    if m in cohort:
        ax.axhline(cohort[m]["p50"], ls="--", c="k", lw=0.8)
    ax.set_title(m)
plt.tight_layout()

In [ ]:
# The fitness axis: resting HR vs steps, coloured by gold class
fig, ax = plt.subplots(figsize=(6, 4.5))
for c in ws.core_classes():
    sub = df[df.gold_class == c]
    ax.scatter(sub["mean_steps"], sub["mean_rhr"], label=f"{c} (n={len(sub)})", c=CLASS_COLOURS[c], s=30)
ax.axvline(cohort["steps"]["p50"], ls="--", c="k", lw=0.8)
ax.axhline(cohort["rhr"]["p50"], ls="--", c="k", lw=0.8)
ax.set_xlabel("28-day mean steps")
ax.set_ylabel("28-day mean resting HR")
ax.legend()
ax.set_title("Fitness axis (dashed: cohort medians)")
print("questions with no steps in the window:", int((df["n_steps"] == 0).sum()),
      "| with no resting HR:", int((df["n_rhr"] == 0).sum()))

### 2d. Blood markers by class

The lab axis. `n_lab_flags` counts adverse flags with common clinical cut-offs (`ws.lab_flag_thresholds()`): an
approximation, since the benchmark's own rule is not published.

In [ ]:
blood_cols = sorted(c for c in df.columns if c.startswith("blood_"))
availability = df[blood_cols].notna().mean().sort_values()
print("share of questions with each marker:")
display(availability.rename(lambda c: c.replace("blood_", "")).to_frame("available").T.round(2))

display(pd.crosstab(df["gold_class"], df["n_lab_flags"], margins=True))

markers = ["blood_hgba1c", "blood_glu", "blood_ins", "blood_tg", "blood_hdl", "blood_hscrp"]
fig, axes = plt.subplots(1, len(markers), figsize=(18, 3.2))
for ax, m in zip(axes, markers):
    data = [df.loc[df.gold_class == c, m].dropna() for c in ws.core_classes()]
    ax.boxplot(data, tick_labels=ws.core_classes())
    ax.set_title(m.replace("blood_", ""))
plt.tight_layout()

### 2e. Missing values

Share of the 29 window days on which each metric has a value, per question (rows sorted by gold class). Real
records are sparse: whole metrics can be missing for a user, and some days have no reading at all.

In [ ]:
order = df.sort_values(["gold_class", "user_id"]).index
coverage = pd.DataFrame({m: df.loc[order, f"n_{m}"] / 29 for m in ws.all_metrics()})
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [3, 1]})
im = axes[0].imshow(coverage.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[0].set_xticks(range(len(coverage.columns)), coverage.columns, rotation=60, ha="right")
bounds = df.loc[order, "gold_class"].values
ticks = [i for i in range(len(bounds)) if i == 0 or bounds[i] != bounds[i - 1]]
axes[0].set_yticks(ticks, [bounds[i] for i in ticks])
axes[0].set_title("share of window days with a value (rows: questions)")
plt.colorbar(im, ax=axes[0])
coverage.mean().sort_values().plot.barh(ax=axes[1], color="grey")
axes[1].set_title("mean coverage per metric")
plt.tight_layout()

print("metrics entirely missing in a question's window (number of questions):")
display((coverage == 0).sum().sort_values(ascending=False).to_frame("questions").T)
print("blood markers missing (number of questions):")
display(df[blood_cols].isna().sum().rename(lambda c: c.replace("blood_", "")).to_frame("questions").T)

## 3. Split each question into private views per device or site

Each holder is an agent with private data. The default assignment (`ws.default_holder_table()`) can be edited:

In [ ]:
holders = ws.default_holder_table()
display(pd.DataFrame({h: {"metrics": ", ".join(s["metrics"]) or "-", "context": ", ".join(s["context"]) or "-"}
                      for h, s in holders.items()}).T)

splits = []
for q in questions:
    s = ws.split_question(q, raw, holders, window_days=28, history_days=0)
    ws.check_partition(q, s, raw)  # every value in exactly one agent
    splits.append(s)
print(f"split {len(splits)} questions; every value is held by exactly one agent")

In [ ]:
# How much each agent holds, per question
rows = []
for s in splits:
    for name, agent in s["agents"].items():
        n_values = sum(len(r) - 1 for r in agent["rows"]) + sum(len(v) for v in agent["context"].values())
        rows.append({"id": s["id"], "gold_class": s["gold_class"], "agent": name, "days": len(agent["rows"]),
                     "values": n_values, "empty": n_values == 0})
holdings = pd.DataFrame(rows)
display(holdings.groupby("agent").agg(questions=("id", "count"), empty_questions=("empty", "sum"),
                                      median_days=("days", "median"), median_values=("values", "median")))

# Per agent and metric: share of questions where the agent has at least one reading
share = {}
for name, spec in holders.items():
    for m in spec["metrics"]:
        share[(name, m)] = np.mean([any(m in r for r in s["agents"][name]["rows"]) for s in splits])
display(pd.Series(share).rename("share of questions with data").to_frame().T.round(2))

## 4. Examples of the split

`show_split` prints the shared part (query, options with class codes, gold), then each agent's private
information and a per-metric summary.

In [ ]:
def show_split(split: dict, max_rows: int = 4) -> None:
    """Print one split question: shared part, gold, then every agent's private view and summary."""
    print("=" * 100)
    print(f"{split['id']}   user {split['user_id']}   window {split['window_days']} days ending {split['end_date']}")
    print("QUERY:", split["shared"]["query"])
    for letter, text in split["shared"]["options"].items():
        mark = "*" if letter == split["gold_letter"] else " "
        print(f"  {letter}{mark} [{split['option_classes'][letter]:4s}] {text[:95]}")
    print(f"GOLD: {split['gold_letter']} = {split['gold_class']} ({ws.phenotype_class_names()[split['gold_class']]})")
    print("SHARED cohort reference medians:",
          {m: v["p50"] for m, v in split["shared"]["cohort_reference"].items()})
    for agent in split["agents"].values():
        print()
        print(ws.render_agent_text(agent, max_rows=max_rows))
        summary = pd.DataFrame(ws.agent_summary(agent)).T
        if len(summary):
            display(summary.round(2))


by_id = {s["id"]: s for s in splits}
show_split(by_id["Lit_user_66_2023-01-07_3"])   # gold D-LF: low fitness, good labs; the phone has no steps

In [ ]:
show_split(by_id["Lit_user_25_2023-12-20_3"])   # gold CA: low fitness, bad labs; phone and watch agree

In [ ]:
# One random example of each gold class
rng = np.random.default_rng(0)
for c in ws.core_classes():
    candidates = [s for s in splits if s["gold_class"] == c]
    show_split(candidates[rng.integers(len(candidates))], max_rows=2)

## 5. Random forest: how separable are the four classes, and which features matter?

Features per question: the 28-day **mean** of each wearable metric, the blood markers, HOMA-IR
(fasting glucose x insulin / 405) and age, sex and BMI. Missing values are median-imputed inside each fold.

Evaluation: 5-fold `StratifiedGroupKFold` **grouped by user** (a user can have several questions, so random folds
would leak), repeated over 10 seeds. D-FA has only 3 questions, so its per-class numbers are very noisy; read
balanced accuracy and macro-F1 next to accuracy.

Importances are **permutation importances on the held-out folds** (drop in accuracy when one feature is shuffled),
which is less biased towards high-cardinality features than the impurity importance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text


def model_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Numeric features: window means, blood markers, HOMA-IR, age, sex (1 = male) and BMI."""
    cols = [c for c in df.columns if c.startswith("mean_") or c.startswith("blood_")] + ["age", "bmi"]
    out = df[cols].astype(float).copy()
    out["sex_m"] = (df["sex"] == "m").astype(float)
    out["homa_ir"] = df["blood_glu"] * df["blood_ins"] / 405
    return out


def agent_feature_groups(holders: dict, columns: list[str]) -> dict[str, list[str]]:
    """Feature columns each agent holds, following the holder table of section 3."""
    groups = {}
    for name, spec in holders.items():
        cols = [f"mean_{m}" for m in spec["metrics"] if f"mean_{m}" in columns]
        if "demographics" in spec["context"]:
            cols += ["age", "sex_m", "bmi"]
        if "blood_panel" in spec["context"]:
            cols += [c for c in columns if c.startswith("blood_")] + ["homa_ir"]
        markers = spec.get("blood_markers", [])
        cols += [f"blood_{m}" for m in markers if f"blood_{m}" in columns]
        if {"glu", "ins"} <= set(markers):
            cols.append("homa_ir")
        if cols:
            groups[name] = cols
    return groups


X_all = model_frame(df)
y = df["gold_class"].to_numpy()
user_groups = df["user_id"].to_numpy()
feature_groups = agent_feature_groups(holders, list(X_all.columns))
for name, cols in feature_groups.items():
    print(f"{name:7s} {len(cols):2d} features: {', '.join(cols)}")
print("\nshare missing per feature (top 10):")
display(X_all.isna().mean().sort_values(ascending=False).head(10).round(2).to_frame("missing").T)

In [ ]:
def cv_evaluate(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, seeds: int = 10, importance_seeds: int = 3,
                n_trees: int = 300) -> dict:
    """Grouped, stratified 5-fold CV repeated over seeds: accuracy, balanced accuracy, macro-F1, pooled confusion
    matrix and held-out permutation importances (averaged over the first `importance_seeds` seeds)."""
    labels = ws.core_classes()
    Xv = X.to_numpy()
    accs, baccs, f1s, imps = [], [], [], []
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for seed in range(seeds):
        pred = np.empty(len(y), dtype=object)
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr, te in cv.split(Xv, y, groups):
            model = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                                  RandomForestClassifier(n_estimators=n_trees, min_samples_leaf=2,
                                                         class_weight="balanced", random_state=seed, n_jobs=-1))
            model.fit(Xv[tr], y[tr])
            pred[te] = model.predict(Xv[te])
            if seed < importance_seeds:
                pi = permutation_importance(model, Xv[te], y[te], n_repeats=10, random_state=seed, scoring="accuracy")
                imps.append(pi.importances_mean)
        accs.append(accuracy_score(y, pred))
        baccs.append(balanced_accuracy_score(y, pred))
        f1s.append(f1_score(y, pred, average="macro"))
        cm += confusion_matrix(y, pred, labels=labels)
    importance = pd.Series(np.mean(imps, axis=0), index=X.columns) if imps else None
    return {"acc": np.mean(accs), "acc_sd": np.std(accs), "bacc": np.mean(baccs), "f1": np.mean(f1s), "cm": cm,
            "importance": importance}


res_all = cv_evaluate(X_all, y, user_groups)
print(f"all features: accuracy {res_all['acc']:.3f} +/- {res_all['acc_sd']:.3f} | "
      f"balanced {res_all['bacc']:.3f} | macro-F1 {res_all['f1']:.3f}")
print(f"majority class (CH): {np.mean(y == 'CH'):.3f} | chance (4 classes): 0.250")

In [ ]:
# Confusion matrix (pooled over seeds, rows normalised) and the two axes separately
labels = ws.core_classes()
cm = res_all["cm"]
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4), labels)
ax.set_yticks(range(4), labels)
ax.set_xlabel("predicted")
ax.set_ylabel("gold")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_title("RF, all features (row-normalised)")
plt.tight_layout()
plt.show()

fit_classes, good_lab_classes = {"CH", "D-FA"}, {"CH", "D-LF"}
idx = {c: i for i, c in enumerate(labels)}
fit_ok = sum(cm[idx[g], idx[p]] for g in labels for p in labels if (g in fit_classes) == (p in fit_classes))
lab_ok = sum(cm[idx[g], idx[p]] for g in labels for p in labels if (g in good_lab_classes) == (p in good_lab_classes))
print(f"fitness-axis accuracy {fit_ok / cm.sum():.3f} | lab-axis accuracy {lab_ok / cm.sum():.3f}")

In [ ]:
# Top permutation importances, coloured by the agent that holds the feature
owner = {c: name for name, cols in feature_groups.items() for c in cols}
agent_colours = {"watch": "#264653", "phone": "#2a9d8f", "ring": "#e9c46a", "clinic": "#e76f51"}
top = res_all["importance"].sort_values(ascending=False).head(20)[::-1]
fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(top.index, top.values, color=[agent_colours.get(owner.get(c), "grey") for c in top.index])
ax.set_xlabel("mean drop in held-out accuracy when shuffled")
ax.set_title("Permutation importance (RF, all features)")
for name, colour in agent_colours.items():
    ax.barh([], [], color=colour, label=name)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
display(res_all["importance"].sort_values(ascending=False).round(4).to_frame("importance").T)

### 5b. Ablations: what can each agent reach alone, and does a small feature set lose anything?

`prompt subset` is the candidate set for the prompts (clinically motivated, see the prompt-template doc):
resting HR, VO2max, HRV, exercise hours (watch); steps, age, sex, BMI (phone); glucose, HbA1c, insulin,
HOMA-IR (glycaemic clinic); triglycerides, HDL (lipid clinic). The ring is left out of the subset.

In [ ]:
prompt_subset = ["mean_rhr", "mean_vo2_max", "mean_hrv", "mean_exercise_hours", "mean_steps", "age", "sex_m", "bmi",
                 "blood_glu", "blood_hgba1c", "blood_ins", "homa_ir", "blood_tg", "blood_hdl"]
top_k = list(res_all["importance"].sort_values(ascending=False).index)
wearable = [c for name in ("watch", "phone", "ring") for c in feature_groups.get(name, [])]
feature_sets = {"all": list(X_all.columns), "prompt subset": prompt_subset, "top-5 (importance)": top_k[:5],
                "top-10 (importance)": top_k[:10], "wearable + phone (no blood)": wearable,
                **{f"{name} only": cols for name, cols in feature_groups.items()}}
rows = []
for name, cols in feature_sets.items():
    r = res_all if name == "all" else cv_evaluate(X_all[cols], y, user_groups, importance_seeds=0)
    rows.append({"feature set": name, "n": len(cols), "accuracy": r["acc"], "sd": r["acc_sd"],
                 "balanced acc": r["bacc"], "macro-F1": r["f1"]})
ablation = pd.DataFrame(rows).set_index("feature set").round(3)
display(ablation)
print("note: top-k sets are chosen on the same data they are scored on, so they are optimistic")

### 5c. How were the labels made? Shallow trees per axis

If a depth-2 tree reproduces an axis exactly, the gold labels come from a threshold rule. Then the useful prompt
content is the variables in that rule, and the expected accuracy depends on how close the prompt's clinical
cut-offs are to the rule's cut-offs.

In [ ]:
axes = {"fit (CH or D-FA)": (np.isin(y, ["CH", "D-FA"]), ["mean_rhr", "mean_steps", "mean_vo2_max", "mean_hrv",
                                                          "mean_exercise_hours", "mean_active_burn"]),
        "favourable labs (CH or D-LF)": (np.isin(y, ["CH", "D-LF"]), ["blood_glu", "blood_hgba1c", "blood_tg",
                                                                        "blood_hdl", "homa_ir", "blood_ins", "bmi"])}
for axis, (target, cols) in axes.items():
    tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_all[cols].fillna(-1), target)
    print(f"{axis}: training accuracy {tree.score(X_all[cols].fillna(-1), target):.3f}")
    print(export_text(tree, feature_names=cols))

### 5d. Clinical-guideline rule: what a model that follows the prompt's clinical key would score

Fitness axis: fit if the 28-day mean resting HR is below a reference (cohort median, or cohort 75th percentile).
Metabolic axis: adverse if any guideline flag is set: fasting glucose >= 100 mg/dL or HbA1c >= 5.7 % (ADA
prediabetes), triglycerides >= 150 mg/dL or HDL < 40 (men) / < 50 (women) mg/dL (harmonised metabolic-syndrome
criteria), HOMA-IR >= 2.5. No training: this is an estimate of the ceiling for a model that reads the clinical key
correctly.

In [ ]:
cohort_rhr = raw["cohort_reference"]["rhr"]


def guideline_adverse(row: pd.Series) -> bool:
    """Any guideline flag on the metabolic panel (missing markers count as not flagged)."""
    hdl_cut = 40 if row["sex_m"] == 1 else 50
    flags = [row["blood_glu"] >= 100, row["blood_hgba1c"] >= 5.7, row["blood_tg"] >= 150, row["blood_hdl"] < hdl_cut,
             row["homa_ir"] >= 2.5]
    return any(bool(f) for f in flags if not pd.isna(f))


adverse = X_all.apply(guideline_adverse, axis=1).to_numpy()
gold_adverse = np.isin(y, ["CA", "D-FA"])
gold_fit = np.isin(y, ["CH", "D-FA"])
print(f"metabolic axis, guideline flags vs gold: {np.mean(adverse == gold_adverse):.3f}")
display(pd.crosstab(pd.Series(gold_adverse, name="gold adverse"), pd.Series(adverse, name="guideline adverse")))
for ref in ("p50", "p75"):
    fit = (X_all["mean_rhr"] < cohort_rhr[ref]).to_numpy()
    pred = np.where(fit & ~adverse, "CH", np.where(fit & adverse, "D-FA", np.where(adverse, "CA", "D-LF")))
    print(f"RHR < cohort {ref} ({cohort_rhr[ref]} bpm): fitness axis {np.mean(fit == gold_fit):.3f} | "
          f"4-class {accuracy_score(y, pred):.3f} | balanced {balanced_accuracy_score(y, pred):.3f}")

### Reading section 5

- RF accuracy is an **upper reference** for what the agents can reach from summary statistics; an LLM that reads
  the same numbers through a clinical key should land near the guideline-rule score in 5d, not above the RF.
- The per-agent rows in 5b show what each agent can decide alone, which is what the conformal sets will reflect:
  the watch should settle the fitness axis, the clinic the metabolic axis, the ring neither.

## Notes to fill in while exploring

- Which metrics separate the fitness axis best (resting HR, steps, VO2max)? Where do the classes overlap?
- How often is an agent empty for a question (no readings in the window)? Which classes does that affect?
- Does the lab-flag count separate CH/D-LF from CA/D-FA in every question?
- Should HRV sit on the ring or the watch? (edit `holders` in section 3 and re-run)